# Quiz VaR y GBM | Exxon Mobil (XOM)
**Ingeniería Financiera (IIND-4414) | Universidad de los Andes**

Daniel Benavides

## 1. Carga de librerías

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd

# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Análisis técnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm, jarque_bera
from scipy import stats

# ARCH model
from arch import arch_model

import warnings
warnings.filterwarnings("ignore")

In [ ]:
plt.rcParams['font.family'] = 'Arial'

## 2. Obtención de datos históricos

In [ ]:
# Descargamos datos de la acción asignada
ticker = "XOM"
start_date = "2021-01-01"
end_date = "2026-10-22"
df = yfin.download(ticker, start=start_date, end=end_date, multi_level_index=False)
df

In [ ]:
mpf.plot(df, type='candle', volume=True, figratio=(19,8), style='yahoo', title='Exxon Mobil (XOM)')

## 3. Retornos
$$
r_{t} = ln \left( \frac{S_t}{S_{t-1}} \right ) = ln \left( S_{t} \right) - ln \left( S_{t-1} \right)
$$

In [ ]:
df['Log Returns'] = np.log(df['Close']) - np.log(df['Close'].shift(1))
df = df.dropna()
df

In [ ]:
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'blue')
plt.title('Retornos Logarítmicos')
plt.xlabel('Fecha')
plt.show()

### 3.1. Jarque-Bera test

In [ ]:
jb_value, p_value = jarque_bera(df['Log Returns'])
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

### 3.2. Ljung-Box test

In [ ]:
lags = 5
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns'], lags)
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

### 3.3. ARCH effects: Ljung-Box test

In [ ]:
df['Log Returns^2'] = df['Log Returns']**2
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns^2'], lags)
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

## Punto 1. VaR - Simulación Histórica

In [ ]:
confidence_level = 0.95
window_size = 252

In [ ]:
df['VaR-SH'] = np.nan
for i in range(window_size, len(df)):
    returns = df['Log Returns'].iloc[i-window_size:i]
    sorted_returns = np.sort(returns)
    df.iloc[i, df.columns.get_loc('VaR-SH')] = np.percentile(sorted_returns, (1 - confidence_level)*100)
# VaR - Simulación Histórica
df['VaR-SH']

In [ ]:
df['VaR-SH'].iloc[-1]

In [ ]:
plt.figure(figsize=(15,8))
plt.plot(df['VaR-SH'], color = 'red')
plt.title('VaR - Simulación Histórica')
plt.xlabel('Fecha')
plt.show()

In [ ]:
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'blue')
plt.plot(df['VaR-SH'], color = 'red')
plt.title('Retornos Logarítmicos & VaR - Simulación Histórica')
plt.xlabel('Fecha')
plt.show()

### Backtesting: VaR - Simulación Histórica

In [ ]:
violations_sh = (df['Log Returns'] < df['VaR-SH']).sum()
total_obs_sh  = df['VaR-SH'].dropna().shape[0]

print(f"=== Backtesting VaR Simulación Histórica ({int(confidence_level*100)}%) ===")
print(f"Observaciones totales : {total_obs_sh:,}")
print(f"Violaciones VaR-SH: {violations_sh} ({100*violations_sh/total_obs_sh:.2f}%)  | esperado: {100*(1-confidence_level):.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(df['Log Returns'], color='#2c3e50', linewidth=0.8, alpha=0.8, label='Retornos Log')
ax.plot(df['VaR-SH'], color='red', linewidth=1.3, label=f'VaR-SH ({int(confidence_level*100)}%)')

mask_viol = df['Log Returns'] < df['VaR-SH']
ax.scatter(df.index[mask_viol], df['Log Returns'][mask_viol],
           color='red', s=20, zorder=5, label=f'Sobrepasadas: {mask_viol.sum()}')

ax.set_title(f'Backtesting VaR Simulación Histórica ({int(confidence_level*100)}%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Retorno Log')
ax.legend(fontsize=10)
ax.set_facecolor('#f8f9fa')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Punto 2. VaR - Paramétrico Normal con GARCH(1,1)
$ \sigma_{t}^{2} = \alpha_{0} + \sum^{q}_{j=1} \alpha_j ϵ_{t-j}^{2} +  \sum^{p}_{i=1} \beta_i \sigma_{t-i}^{2} $

In [ ]:
model = arch_model(df['Log Returns']*100, mean='Zero', vol='GARCH', q=1, p=1, rescale=False)
results = model.fit()
print(results.summary())

In [ ]:
params = pd.DataFrame(results.params)
params

In [ ]:
df['Volatility'] = results.conditional_volatility/100*math.sqrt(252)
plt.figure(figsize=(15,8))
plt.plot(df['Volatility'], color = 'orange')
plt.title('Volatility - GARCH(1,1)')
plt.xlabel('Fecha')
plt.ylabel('Volatility')
plt.show()

In [ ]:
print("El último valor de volatilidad es", round(df['Volatility'].iloc[-1]*100,4),"%")

In [ ]:
df['VaR'] = np.nan
for i in range(0, len(df)):
    returns = df['Log Returns'].iloc[0:i]
    mean_val = returns.mean() if not returns.empty else 0
    df.iloc[i, df.columns.get_loc('VaR')] = mean_val + norm.ppf(1 - confidence_level) * df.iloc[i, df.columns.get_loc('Volatility')] * np.sqrt(1/252)
# VaR - Paramétrico (media + GARCH)
df['VaR']

In [ ]:
df['VaR_2'] = np.nan
for i in range(0, len(df)):
    df.iloc[i, df.columns.get_loc('VaR_2')] = norm.ppf(1 - confidence_level) * df.iloc[i, df.columns.get_loc('Volatility')] * np.sqrt(1/252)
# VaR - Paramétrico (GARCH)
df['VaR_2']

In [ ]:
print(f"VaR Paramétrico 1d (media+GARCH)  [{int(confidence_level*100)}%]: {float(df['VaR'].iloc[-1])*100:.4f} %")
print(f"VaR Paramétrico 1d (GARCH)        [{int(confidence_level*100)}%]: {float(df['VaR_2'].iloc[-1])*100:.4f} %")

In [ ]:
plt.figure(figsize=(15,8))
plt.plot(df['VaR'], color = 'red')
plt.title('VaR - Paramétrico')
plt.xlabel('Fecha')
plt.show()

In [ ]:
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'blue')
plt.plot(df['VaR'], color = 'red')
plt.plot(df['VaR_2'], color = 'orange')
plt.title('Retornos Logarítmicos & VaR - Paramétrico Normal')
plt.xlabel('Fecha')
plt.show()

### Backtesting: VaR - Paramétrico

In [ ]:
violations   = (df['Log Returns'] < df['VaR']).sum()
violations_2 = (df['Log Returns'] < df['VaR_2']).sum()
total_obs    = df['Log Returns'].dropna().shape[0]

print(f"=== Backtesting VaR Paramétrico ({int(confidence_level*100)}%) ===")
print(f"Observaciones totales : {total_obs:,}")
print(f"Violaciones VaR 1 (media+GARCH): {violations} ({100*violations/total_obs:.2f}%)  | esperado: {100*(1-confidence_level):.2f}%")
print(f"Violaciones VaR 2 (GARCH):       {violations_2} ({100*violations_2/total_obs:.2f}%)  | esperado: {100*(1-confidence_level):.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(df['Log Returns'], color='#2c3e50', linewidth=0.8, alpha=0.8, label='Retornos Log')
ax.plot(df['VaR_2'], color='red', linewidth=1.3, label=f'VaR GARCH 1d ({int(confidence_level*100)}%)')

mask_viol = df['Log Returns'] < df['VaR_2']
ax.scatter(df.index[mask_viol], df['Log Returns'][mask_viol],
           color='red', s=20, zorder=5, label=f'Sobrepasadas: {mask_viol.sum()}')

ax.set_title(f'Backtesting VaR Paramétrico ({int(confidence_level*100)}%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Retorno Log')
ax.legend(fontsize=10)
ax.set_facecolor('#f8f9fa')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Geometric Brownian Motion (GBM) - Parámetros

In [ ]:
sigma_garch = df['Volatility'].iloc[-1]
sigma_garch

In [ ]:
r_avg = np.mean(df['Log Returns'])*252
r_avg

In [ ]:
# Define the parameters of the stochastic process
T = 3/12        # Time horizon
N = int(252*T)  # Number of time steps
dt = T / N      # Time step size

sigma = sigma_garch                    # Volatility of the process
mu = r_avg + (1/2)*sigma_garch**2      # Mean of the process
div = 0                                # Dividend yield
r = np.log(1 + 0.0388)                 # r: risk-free interest rate (SOFR 21-oct-2026, continua)

# Define the initial condition
S0 = df['Close'].iloc[-1]
K = S0                                 # K: strike price (at-the-money)

print("S0 =", S0)
print("K =", K)
print("T =", T, "| N =", N, "| dt =", dt)
print("sigma =", sigma)
print("mu =", mu)
print("r =", r)
print("div =", div)

## Punto 3. GBM - Probabilidades reales

In [ ]:
def gbm_paths(S0, mu, sigma, T, N, num_paths):
    dt = T/N
    paths = np.zeros((num_paths, N+1))
    paths[:, 0] = S0

    for i in range(num_paths):
        for j in range(1, N+1):
            drift = (mu - 0.5 * sigma**2) * dt
            diffusion = sigma * np.sqrt(dt) * np.random.normal()
            paths[i, j] = paths[i, j-1] * np.exp(drift + diffusion)

    return paths

In [ ]:
np.random.seed(42)
num_paths = 10000

paths = gbm_paths(S0, mu, sigma, T, N, num_paths)

plt.figure(figsize=(10, 6))
plt.plot(paths.T)
plt.xlabel('Time Steps')
plt.ylabel('Stock Price')
plt.title('Geometric Brownian Motion')
plt.show()

### Call option price

In [ ]:
# calculate the call option payoff at maturity
payoff = np.maximum(paths[:, -1] - K, 0)
# calculate the call option price
call_price = np.exp(-r * T) * np.mean(payoff)
call_se = np.exp(-r * T) * np.std(payoff) / np.sqrt(num_paths)
print(f"Call: {call_price:.4f}  | IC 95%: [{call_price - 1.96*call_se:.4f}, {call_price + 1.96*call_se:.4f}]")

### Put option price

In [ ]:
# calculate the put option payoff at maturity
payoff = np.maximum(K - paths[:, -1], 0)
# calculate the put option price
put_price = np.exp(-r * T) * np.mean(payoff)
put_se = np.exp(-r * T) * np.std(payoff) / np.sqrt(num_paths)
print(f"Put: {put_price:.4f}  | IC 95%: [{put_price - 1.96*put_se:.4f}, {put_price + 1.96*put_se:.4f}]")

## Punto 4. GBM - Risk neutral

In [ ]:
def gbm_paths_RN(S0, mu, sigma, T, N, num_paths):
    dt = T/N
    paths = np.zeros((num_paths, N+1))
    paths[:, 0] = S0

    for i in range(num_paths):
        for j in range(1, N+1):
            drift = ((r-div) - 0.5 * sigma**2) * dt
            diffusion = sigma * np.sqrt(dt) * np.random.normal()
            paths[i, j] = paths[i, j-1] * np.exp(drift + diffusion)

    return paths

In [ ]:
np.random.seed(42)

paths_RN = gbm_paths_RN(S0, mu, sigma, T, N, num_paths)

plt.figure(figsize=(10, 6))
plt.plot(paths_RN.T)
plt.xlabel('Time Steps')
plt.ylabel('Stock Price')
plt.title('Geometric Brownian Motion - Risk Neutral')
plt.show()

### Call option price - RN

In [ ]:
# calculate the call option payoff at maturity
payoff = np.maximum(paths_RN[:, -1] - K, 0)
# calculate the call option price
call_price_RN = np.exp(-r * T) * np.mean(payoff)
call_se_RN = np.exp(-r * T) * np.std(payoff) / np.sqrt(num_paths)
print(f"Call RN: {call_price_RN:.4f}  | IC 95%: [{call_price_RN - 1.96*call_se_RN:.4f}, {call_price_RN + 1.96*call_se_RN:.4f}]")

### Put option price - RN

In [ ]:
# calculate the put option payoff at maturity
payoff = np.maximum(K - paths_RN[:, -1], 0)
# calculate the put option price
put_price_RN = np.exp(-r * T) * np.mean(payoff)
put_se_RN = np.exp(-r * T) * np.std(payoff) / np.sqrt(num_paths)
print(f"Put RN: {put_price_RN:.4f}  | IC 95%: [{put_price_RN - 1.96*put_se_RN:.4f}, {put_price_RN + 1.96*put_se_RN:.4f}]")

## Black-Scholes-Merton

In [ ]:
# Black-Scholes-Merton formula for a call option
def black_scholes_merton_call(S, K, T, r, sigma, div):
    d1 = (np.log(S / K) + (r - div + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * np.exp(-div * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

# Calculate the call option price using BSM
bsm_call_price = black_scholes_merton_call(S0, K, T, r, sigma, div)
print("Black-Scholes-Merton Call Option Price:", bsm_call_price)

In [ ]:
# Black-Scholes-Merton formula for a put option
def black_scholes_merton_put(S, K, T, r, sigma, div):
    d1 = (np.log(S / K) + (r - div + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-div * T) * norm.cdf(-d1)
    return put_price

# Calculate the put option price using BSM
bsm_put_price = black_scholes_merton_put(S0, K, T, r, sigma, div)
print("Black-Scholes-Merton Put Option Price:", bsm_put_price)